In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import ScrapeWebsiteTool, FileWriterTool
from crewai.tools import tool
from ddgs import DDGS

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.3,
        max_tokens=1500,
    )

    @tool("DuckDuckGoSearch")
    def duckduckgo_search(query: str) -> str:
        """Search DuckDuckGo and return titles, snippets and URLs."""
        results = DDGS().text(query, max_results=5)
        return "\n\n".join(f"{r.get('title')}\n{r.get('href')}\n{r.get('body')}" for r in results)

    competitor_input = input("Enter competitor URLs (comma-separated) [default: https://www.notion.so, https://clickup.com]: ").strip()
    if not competitor_input:
        competitor_input = "https://www.notion.so, https://clickup.com"

    site_scraper = Agent(
        role="Website Scraper",
        goal="Extract product offerings and features from competitor sites: {competitors}",
        backstory="Expert at parsing web pages and identifying core value propositions.",
        llm=llm,
        tools=[ScrapeWebsiteTool()],
        verbose=True,
        allow_delegation=False,
    )

    pricing_analyst = Agent(
        role="Pricing Analyst",
        goal="Extract tiers, pricing structures, and limits for {competitors}",
        backstory="Specialist in SaaS pricing models and plan comparisons.",
        llm=llm,
        tools=[ScrapeWebsiteTool()],
        verbose=True,
        allow_delegation=False,
    )

    news_analyst = Agent(
        role="News Analyst",
        goal="Find recent product announcements and updates regarding {competitors}",
        backstory="Market intelligence researcher monitoring tech launches and updates.",
        llm=llm,
        tools=[duckduckgo_search],
        verbose=True,
        allow_delegation=False,
    )

    strategist = Agent(
        role="Strategy Consultant",
        goal="Synthesize competitor findings into a SWOT matrix and strategic recommendations",
        backstory="Executive advisor providing actionable market insights.",
        llm=llm,
        tools=[FileWriterTool()],
        verbose=True,
        allow_delegation=False,
    )

    scrape_task = Task(
        description="Visit competitor websites: {competitors}. Summarize product positioning and primary offerings.",
        expected_output="A structured summary per competitor.",
    )

    pricing_task = Task(
        description="Find pricing tiers, billing cycles, and feature limits for {competitors}.",
        expected_output="A comparative pricing overview per competitor.",
    )

    news_task = Task(
        description="Search for the latest news and feature releases for {competitors}.",
        expected_output="3 recent updates per competitor with citations.",
    )

    strategy_task = Task(
        description="Using gathered data, create a SWOT matrix and 3 strategic recommendations. Save to output/report.md.",
        expected_output="A comprehensive Markdown competitor analysis report.",
        context=[scrape_task, pricing_task, news_task],
    )

    crew = Crew(
        agents=[site_scraper, pricing_analyst, news_analyst, strategist],
        tasks=[scrape_task, pricing_task, news_task, strategy_task],
        process=Process.hierarchical,
        manager_llm=llm,
        verbose=True,
    )

    result = crew.kickoff(inputs={"competitors": competitor_input})
    print("\n--- Final Competitor Analysis Report ---\n")
    print(result.raw)

except Exception as e:
    print("Error:", e)
